# Slide figures for the BOLDsωimsuite talk

This notebook makes the five simulation figures on slides 8 and 11–14, from scratch, with BOLDsωimsuite.

**How to run it**
1. Use the same Python 3.10 kernel you used for Jack's notebook (**Kernel → Change kernel → BOLDswimsuite (py3.10)**, or whatever you named the `bold` environment).
2. Run the cells from top to bottom with **Shift + Enter**, or **Run → Run All Cells**.
3. The whole thing takes about 10–15 minutes. Almost all of it is cell 2 (the simulations).

**What it makes**
- `out/`: 96 small simulation files (about 3 MB in total)
- `figs/`: the five figures as PNG files, plus `summary.json` with the numbers behind them

| Cell | What it does | Slide |
| --- | --- | --- |
| 0 | Checks the kernel and the files | |
| 1 | Install check (same numbers as Jack's) | |
| 2 | Runs the 96 simulations | |
| 3 | Loads the results and does the dictionary matching | |
| 4 | Field maps: own direction per vessel vs 3B0 | 8 |
| 5 | What SO2 and vessel size do to the GESFIDE signal | 11 |
| 6 | Decay rate vs vessel size (Boxerman plot) | 12 |
| 7 | Mini fingerprinting test: radius | 13 |
| 8 | Mini fingerprinting test: SO2 | 14 |
| 9 | Compares your numbers with mine | |

You can run it again at any time. Simulations that are already in `out/` are skipped, so a second run takes about a minute.

## Cell 0 · Check the kernel and the files

If this cell stops with a message, read the message: it says what to fix.

In [ ]:
import os, sys, glob, json, time
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

print('Python', sys.version.split()[0])
if sys.version_info[:2] != (3, 10):
    raise RuntimeError('Wrong kernel. Choose the Python 3.10 kernel you used for Jack\'s notebook: Kernel -> Change kernel.')

needed = ['bss_common.py', 'run_play.py', 'compare_to_mine.py', 'reference_summary.json']
missing = [f for f in needed if not os.path.exists(f)]
if missing:
    raise RuntimeError(f'Cannot find {missing} in {os.getcwd()}. Open this notebook from inside the bold-slide-figures folder.')

import boldswimsuite
import bss_common as bc
import run_play
print('Working folder:', os.getcwd())
print('BOLDswimsuite imported; numpy', np.__version__, '(needs to be below 2)')
print('Echo times (ms):', bc.TE[:3], '...', bc.TE[-2:], f'-> {len(bc.TE)} echoes, spin echo at {bc.TE[bc.SE_IDX]:.0f} ms')

## Cell 1 · Install check

The same check as `check_install.py` in Jack's setup guide. You should see exactly:

```
field check: 2.035752e-07 vs 2.035752e-07 -> OK
mini GESFIDE: 32 vessels, S(50 ms) = 0.9334, S(100 ms) = 0.9570
```

The first line checks the field next to one vessel against the textbook formula. The second runs a tiny simulation with fixed seeds, so the numbers repeat exactly on any computer. The times in brackets will differ; that is fine.

In [ ]:
from boldswimsuite import BOLDvessel, BOLDgeometry, BOLDspins, BOLDsequence

t = time.time()
c = BOLDvessel.InfiniteCylinder3D(diameter=0.01, theta=np.pi/2, phi=0.0, origin=np.zeros(3), dchi=4.32e-8)
got = c.dBz_EV(np.array([[0., 0., 0.01]]), 3.0)[0]
expected = 3.0 * (4*np.pi*4.32e-8) / 2 * 0.25        # textbook formula at 2 radii from the vessel
print(f'field check: {got:.6e} vs {expected:.6e} -> {"OK" if np.isclose(got, expected) else "FAIL"} (first call compiles: {time.time()-t:.1f} s)')

vox = BOLDgeometry.ContinuousVoxel3D.from_random(CBV=0.02, B0=3.0, ADC=1e-3, labels=['v'], weights={'v': 1},
      diameter_distributions={'v': [0.006]}, dchis={'v': 3e-8}, num_vessels=30, seed=0, progressbar=False)
sp = BOLDspins.Spins3D(num_spins=2000, geometry=vox, dt=0.2, seed=0)
seq = BOLDsequence.SpinSequence(sp, [0, 250], [np.pi/2, np.pi], ['y', 'x'])
t = time.time(); tot, ev, iv = seq.walk(dt=0.2, num_steps=500, progressbar=False)
print(f'mini GESFIDE: {len(vox.vessels)} vessels, S(50 ms) = {tot[249]:.4f}, S(100 ms) = {tot[499]:.4f} (walk {time.time()-t:.1f} s)')

## Cell 2 · Run the simulations (about 10 minutes)

This is the slow cell. It runs **96 simulations**, one per line of output, on the lab's 38-echo GESFIDE timing:

| Recipe | What it is | Voxels (seeds) |
| --- | --- | --- |
| `3d` | 3D-ANA-MC, Berman's reference: random cylinders in 3D, exact field, Monte Carlo water | 4 (1 and 2 act as the "measured" signals, 3 and 4 as a 3D dictionary) |
| `random` | 2D, each vessel gets its own random field direction (Berman's best 2D recipe) | 2 |
| `3b0` | Same 2D vessels, field averaged over three directions, as in MrVox | 2 |

Each recipe is run at 12 vessel radii from 1 to 50 µm (a 5 µm voxel scaled up or down), at 3% blood volume, with 10,000 water spins and T2 decay turned off. Each run is stored for 66 oxygenation values (SO2 30–95%) at once: oxygenation only scales the field, so one run covers every SO2 (the trick in Jack's notebook, section 7).

The 3D runs come first and take about 30–40 s each; the 2D runs at the end take a few seconds. If you stop the cell (■ button), run it again later: finished runs are skipped.

In [ ]:
N_CORES = min(4, os.cpu_count() or 1)   # how many simulations run side by side; change it if you like
OUT_DIR = 'out'
os.makedirs(OUT_DIR, exist_ok=True)

import multiprocessing
todo = sorted(run_play.jobs(), key=lambda job: (job[0] != '3d', -job[1]))   # slow 3D runs first
print(f'{len(todo)} simulations on {N_CORES} cores, saved in the folder "{OUT_DIR}"', flush=True)
t0 = time.time()
with multiprocessing.get_context('spawn').Pool(N_CORES) as pool:   # 'spawn' = how Mac and Windows start workers
    for i, (path, seconds) in enumerate(pool.imap_unordered(run_play._run, [(job, OUT_DIR) for job in todo]), 1):
        note = 'already done, skipped' if seconds == 0 else f'{seconds:.0f} s'
        print(f'[{i}/{len(todo)}] {os.path.basename(path)}  {note}  (elapsed {time.time()-t0:.0f} s)', flush=True)
print(f'Done: {len(glob.glob(os.path.join(OUT_DIR, "*.npz")))} of 96 runs in "{OUT_DIR}" after {(time.time()-t0)/60:.1f} min.')

## Cell 3 · Load the results and do the dictionary matching

This is the "fingerprinting" part, in miniature:

- The **test signals** play the role of measured data: 3D reference runs (voxels 1 and 2) at true SO2 of 50, 60, 70 and 80%.
- There are **three dictionaries**, built three ways: 3D with new voxels (3 and 4), 2D with its own direction per vessel, and 2D 3B0. Each has 12 radii × 66 SO2 values = 792 entries.
- **Matching** picks the dictionary entry with the largest normalized inner product with the test signal (the usual MRvF matching), over radius and SO2 together.
- It also repeats the matching with added noise at an SNR of 100.

Nothing is drawn yet; the cell prints a short summary and saves `figs/summary.json`.

In [ ]:
FIG_DIR = 'figs'
os.makedirs(FIG_DIR, exist_ok=True)
files = glob.glob(os.path.join(OUT_DIR, '*.npz'))
if len(files) < 96:
    raise RuntimeError(f'Only {len(files)} of 96 runs in "{OUT_DIR}". Run cell 2 until it finishes.')

runs = {}
for p in files:
    z = np.load(p)
    runs[(str(z['approach']), float(z['radius_um']), int(z['seed']))] = z
RADII = sorted({k[1] for k in runs})
radii = np.array(RADII)
SO2 = runs[('3d', 5.0, 1)]['so2']
TE = bc.TE

def mean_signal(approach, seeds, part='tot'):
    'Signal averaged over voxels: array (n_radii, n_SO2, 38 echoes). part = tot (blood + tissue), ev (tissue) or iv (blood).'
    return np.stack([np.mean([runs[(approach, r, s)][part] for s in seeds], 0) for r in RADII])

def match(D, S):
    'Normalized inner product matching. D: (n_radii, n_SO2, 38); S: (..., 38). Returns radius and SO2 indices.'
    Dn = D.reshape(-1, D.shape[-1]); Dn = Dn / np.linalg.norm(Dn, axis=1, keepdims=True)
    Sn = S.reshape(-1, S.shape[-1]); Sn = Sn / np.linalg.norm(Sn, axis=1, keepdims=True)
    best = np.argmax(Sn @ Dn.T, axis=1)
    iR, iS = np.unravel_index(best, D.shape[:2])
    return iR.reshape(S.shape[:-1]), iS.reshape(S.shape[:-1])

ref_all = {p: mean_signal('3d', [1, 2, 3, 4], p) for p in ('tot', 'ev', 'iv')}   # 3D reference, all 4 voxels
test = mean_signal('3d', [1, 2])                                                 # the "measured" signals
dicts = {
    '3D, same recipe as the test': mean_signal('3d', [3, 4]),
    '2D, own field direction per vessel': mean_signal('random', [11, 12]),
    '2D, 3B0 average (MrVox-style)': mean_signal('3b0', [11, 12]),
}

SO2_TRUE = [0.5, 0.6, 0.7, 0.8]
iso = [int(np.argmin(np.abs(SO2 - s))) for s in SO2_TRUE]
i6 = iso[1]                                   # index of SO2 = 60%
results = {}
for name, D in dicts.items():
    iR, iS = match(D, test[:, iso, :])
    results[name] = dict(estR=radii[iR], estSO2=SO2[iS])

rng = np.random.default_rng(0)                # noise test: SNR 100 on the first echo, 200 noise draws
noisy = {}
for name, D in dicts.items():
    S = test[:, i6, :]
    sigma = S[:, :1] / 100
    draws = np.stack([np.abs(S + sigma*(rng.standard_normal(S.shape) + 1j*rng.standard_normal(S.shape))) for _ in range(200)], 1)
    iR, iS = match(D, draws)
    noisy[name] = dict(medR=np.median(radii[iR], 1), medS=np.median(SO2[iS], 1))

def rates(sig):
    'Decay rates (1/s) at SO2 60%: gradient echo at 29.4 ms and spin echo at 100 ms.'
    return (-np.log(sig[:, i6, bc.GE_IDX]) / (TE[bc.GE_IDX]/1000), -np.log(sig[:, i6, bc.SE_IDX]) / (TE[bc.SE_IDX]/1000))
box = {'3D reference': {p: rates(ref_all[p]) for p in ('tot', 'ev', 'iv')}}
for label, a in (('2D, own direction', 'random'), ('2D, 3B0 average', '3b0')):
    box[label] = {p: rates(mean_signal(a, [11, 12], p)) for p in ('tot', 'ev', 'iv')}

summary = {'radii_um': RADII, 'so2_true': SO2_TRUE}
for name, r in results.items():
    err = np.abs(r['estR'] - radii[:, None]) / radii[:, None] * 100
    wrong = r['estR'] != radii[:, None]
    summary[name] = {
        'radius_wrong_at_so2_0.6': int(wrong[:, 1].sum()),
        'radius_mean_abs_err_pct_at_so2_0.6': float(err[:, 1].mean()),
        'radius_max_abs_err_pct_at_so2_0.6': float(err[:, 1].max()),
        'radius_wrong_all_so2': int(wrong.sum()),
        'so2_err_points_at_0.6': [round(float(x), 3) for x in (r['estSO2'][:, 1] - 0.6) * 100],
        'so2_mean_abs_err_points_all': float(np.abs(r['estSO2'] - np.array(SO2_TRUE)[None]).mean() * 100),
        'estR_at_so2_0.6': [float(x) for x in r['estR'][:, 1]],
        'noisy_snr100_median_R': [float(x) for x in noisy[name]['medR']],
        'noisy_snr100_median_SO2': [float(x) for x in noisy[name]['medS']],
    }
summary['boxerman'] = {k: {p: [[round(float(x), 3) for x in v[0]], [round(float(x), 3) for x in v[1]]] for p, v in d.items()} for k, d in box.items()}
summary['iv_fraction'] = {a: float(np.mean([z['iv_fraction'] for k, z in runs.items() if k[0] == a])) for a in ('3d', 'random', '3b0')}

for name in results:
    print(f'{name:36s} radii right at SO2 60%: {12 - summary[name]["radius_wrong_at_so2_0.6"]} of 12')

### Figure style

The same look as the slides: a Times-like serif font, dark ink, one blue and one red. If Liberation Serif is not on your computer, matplotlib uses Times New Roman instead.

In [ ]:
INK, MUTED, HAIR = '#1B1B1F', '#5B5F66', '#D9DADC'
RED, BLUE, GRAY = '#A8322D', '#1F4E79', '#8A8F96'
plt.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Liberation Serif', 'Times New Roman', 'Times', 'DejaVu Serif'],
    'mathtext.fontset': 'stix', 'font.size': 20, 'axes.edgecolor': MUTED, 'axes.linewidth': 1.0,
    'axes.labelcolor': INK, 'xtick.color': MUTED, 'ytick.color': MUTED, 'xtick.labelcolor': INK,
    'ytick.labelcolor': INK, 'xtick.labelsize': 18, 'ytick.labelsize': 18, 'axes.labelsize': 21,
    'axes.titlesize': 22, 'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white', 'figure.dpi': 60,
})
colors = {'3D, same recipe as the test': GRAY, '2D, own field direction per vessel': BLUE, '2D, 3B0 average (MrVox-style)': RED}

def log_radius_axis(a):
    a.set_xscale('log'); a.set_xticks([1, 3, 10, 30]); a.set_xticklabels(['1', '3', '10', '30'])

## Cell 4 · Field maps: own direction per vessel vs 3B0 (slide 8)

The same 2D vessels, with the field computed two ways. On the left, each vessel has its own field direction, so the patterns point different ways and the blood inside each vessel sits at a different offset. On the right (3B0), every pattern points the same way and the blood in every vessel sits at about the same offset, close to zero. That is why the blood signal stops decaying with 3B0.

In [ ]:
base = bc.voxel2d(11)
g = bc.grids2d(base, 512)
fig, ax = plt.subplots(1, 2, figsize=(14, 6.4))
for a, key, title in zip(ax, ('random', '3b0'), ('each vessel, its own field direction', '3B0: one averaged field for all')):
    idx, dbz, _ = g[key]
    im = a.imshow((dbz[:200, :200] * 1e9).T, origin='lower', cmap='RdBu_r', vmin=-60, vmax=60)   # a corner of the voxel, in nT
    a.contour(idx[:200, :200].T > 0, levels=[0.5], colors=INK, linewidths=0.8)                 # vessel edges
    a.set_xticks([]); a.set_yticks([])
    for s in a.spines.values():
        s.set_visible(False)
    a.set_title(title, pad=12)
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02); cb.set_label('field offset (nT)')
fig.savefig(os.path.join(FIG_DIR, 'fieldmaps.png'), dpi=110, bbox_inches='tight')
plt.show()

summary['iv_field_nT_p10_p50_p90'] = {k: [round(float(x), 1) for x in np.percentile(g[k][1][g[k][0] > 0] * 1e9, [10, 50, 90])] for k in ('random', '3b0')}
print('Field inside the vessels, 10th / 50th / 90th percentile (nT):', summary['iv_field_nT_p10_p50_p90'])

## Cell 5 · What the knobs do to the GESFIDE signal (slide 11)

3D reference, T2 decay off. Left: lower oxygenation means more magnetic blood and faster decay. Right: vessel size changes the shape. Large vessels (static field) refocus strongly at the 100 ms spin echo; small vessels hardly refocus, because water diffuses past them. The dashed line is the 180° pulse at 50 ms, the dotted line the spin echo.

In [ ]:
def gap(y):
    'Leave a gap at the 180 pulse so the two halves of the echo train are not joined by a line.'
    k = int(np.searchsorted(TE, 50.0))
    return np.insert(TE, k, np.nan), np.insert(y, k, np.nan)

fig, ax = plt.subplots(1, 2, figsize=(16, 5.6), sharey=True)
for j, s in enumerate([0.5, 0.6, 0.7, 0.8, 0.9]):
    y = ref_all['tot'][RADII.index(5.0), int(np.argmin(np.abs(SO2 - s)))]
    ax[0].plot(*gap(y), '.-', ms=6, lw=2, color=plt.get_cmap('Blues')(0.35 + 0.15*j), label=f'SO$_2$ {int(s*100)}%')
for j, r in enumerate([2.0, 5.0, 10.0, 30.0]):
    y = ref_all['tot'][RADII.index(r), i6]
    ax[1].plot(*gap(y), '.-', ms=6, lw=2, color=plt.get_cmap('Reds')(0.35 + 0.18*j), label=f'R {r:g} µm')
for a in ax:
    a.axvline(50, color=HAIR, lw=1.5, ls='--'); a.axvline(100, color=HAIR, lw=1.5, ls=':'); a.set_xlabel('echo time (ms)')
ax[0].set_ylabel('signal (T2 off)')
ax[0].set_title('Change oxygenation (R 5 µm, CBV 3%)', pad=12)
ax[1].set_title('Change vessel size (SO$_2$ 60%, CBV 3%)', pad=12)
ax[0].legend(fontsize=16, loc='lower left'); ax[1].legend(fontsize=16, loc='lower left')
plt.tight_layout(w_pad=2.5)
fig.savefig(os.path.join(FIG_DIR, 'knobs.png'), dpi=110)
plt.show()

## Cell 6 · Decay rate vs vessel size, the Boxerman plot (slide 12)

How fast the signal decays at a gradient echo (29 ms, left) and at the spin echo (100 ms, right), for each vessel radius. The 2D recipe with its own direction per vessel sits on the 3D reference. 3B0 falls short in the gradient echo, by about a quarter for large vessels and more for small ones, because its blood signal hardly decays. This is Berman's Figure 8C, on our sequence.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5.6))
style = {'3D reference': dict(color=INK, marker='o', ls='-'),
         '2D, own direction': dict(color=BLUE, marker='s', ls='--'),
         '2D, 3B0 average': dict(color=RED, marker='^', ls=':')}
for k, d in box.items():
    ge, se = d['tot']
    ax[0].plot(radii, ge, ms=8, lw=2, mfc='white', mew=2, label=k, **style[k])
    ax[1].plot(radii, se, ms=8, lw=2, mfc='white', mew=2, label=k, **style[k])
for a, title in zip(ax, ['gradient echo, TE 29 ms: ΔR$_2$*', 'spin echo, TE 100 ms: ΔR$_2$']):
    log_radius_axis(a); a.set_xlabel('vessel radius (µm)'); a.set_title(title, pad=12)
ax[0].set_ylabel('relaxation rate (1/s)')
ax[1].legend(fontsize=17)
plt.tight_layout(w_pad=3)
fig.savefig(os.path.join(FIG_DIR, 'boxerman_gesfide.png'), dpi=110)
plt.show()

print('Blood (IV) gradient-echo decay rate, 1/s:  3D reference', np.round(box['3D reference']['iv'][0][[0, 5, 11]], 1),
      '  3B0', np.round(box['2D, 3B0 average']['iv'][0][[0, 5, 11]], 1), '  (at 1, 5 and 50 µm)')

## Cell 7 · Mini fingerprinting test: does each dictionary get the radius right? (slide 13)

The 3D reference signals at SO2 60% are matched against each dictionary. Horizontal: the true radius. Vertical: the radius the matching picked. On the gray line means right. The 3D and 2D own-direction dictionaries miss by at most one grid step; the 3B0 dictionary calls small vessels 7–15 µm.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 5.6), sharex=True, sharey=True)
for a, (name, r) in zip(ax, results.items()):
    a.plot([0.8, 65], [0.8, 65], color=HAIR, lw=2, zorder=1)
    a.plot(radii, r['estR'][:, 1], 'o', ms=11, mfc='white', mec=colors[name], mew=2.4, zorder=3)
    a.set_xscale('log'); a.set_yscale('log')
    a.set_xticks([1, 3, 10, 30]); a.set_xticklabels(['1', '3', '10', '30'])
    a.set_yticks([1, 3, 10, 30]); a.set_yticklabels(['1', '3', '10', '30'])
    a.set_xlim(0.8, 65); a.set_ylim(0.8, 65)
    a.set_title(name, color=INK if colors[name] == GRAY else colors[name], pad=14)
    a.set_xlabel('true radius (µm)')
    n_right = len(radii) - summary[name]['radius_wrong_at_so2_0.6']
    a.text(0.04, 0.95, f'{n_right} of {len(radii)} radii right', transform=a.transAxes, va='top', fontsize=19, color=MUTED)
ax[0].set_ylabel('estimated radius (µm)')
plt.tight_layout(w_pad=2.5)
fig.savefig(os.path.join(FIG_DIR, 'fp_radius.png'), dpi=110)
plt.show()

## Cell 8 · The same matches, oxygenation (slide 14)

True SO2 is 60%. Error in percentage points against the true radius. Both good recipes stay within about 7 points for vessels of 2 µm and up (mostly 2–5 points low). 3B0 reads small vessels up to 20 points too high and large vessels 8–12 points too low.

In [ ]:
fig, a = plt.subplots(figsize=(10, 5.6))
a.axhline(0, color=HAIR, lw=2)
for name, r in results.items():
    e = (r['estSO2'] - np.array(SO2_TRUE)[None]) * 100
    a.plot(radii, e[:, 1], 'o-', ms=9, lw=2, color=colors[name], mfc='white', mew=2.2, label=name)
log_radius_axis(a)
a.set_xlabel('true radius (µm)'); a.set_ylabel('SO$_2$ error (points)')
a.legend(fontsize=17, loc='best')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fp_so2.png'), dpi=110)
plt.show()

with open(os.path.join(FIG_DIR, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=1)
print('Saved:', sorted(os.listdir(FIG_DIR)))

## Cell 9 · Compare your numbers with mine

The simulations use fixed random seeds, so a working install gives the same numbers as my run. Every line should say `match`, and the last line should say that all checks match.

In [ ]:
%run compare_to_mine.py figs

## Done

Your five figures are in the `figs` folder. They are the same images as on slides 8 and 11–14, so you can say "these are my runs."

One sentence for each, if Audrey asks:
- **Slide 8:** 3B0 averages the field over three directions, so the blood in every vessel sits at the same offset and stops dephasing.
- **Slide 11:** less oxygen means faster decay; vessel size changes how much the spin echo brings back.
- **Slide 12:** the 2D recipe with its own direction per vessel matches the 3D reference; 3B0 loses gradient-echo decay.
- **Slide 13:** matched against a 3B0 dictionary, 3D signals give the wrong radius at 9 of 12 sizes.
- **Slide 14:** the wrong radius drags SO2 with it, by up to 20 points.

Caveats to say out loud: one blood volume (3%), T2 off, total signal (blood + tissue), and only 2 voxels × 10,000 spins per dictionary entry. With a tissue-only dictionary the 3B0 bias would be much smaller.